# Project 2: Lagos building footprint mapping

This notebook exports existing Google Research Open Buildings V3 predictions clipped to one Lagos LGA. Earth Engine handles the source data in the cloud; Colab downloads no imagery archive. The default LGA is Ikeja.


In [ ]:
%pip -q install earthengine-api folium
import ee
import folium
import time
from IPython.display import display

EE_PROJECT_ID = input('Earth Engine Google Cloud Project ID: ').strip()
if not EE_PROJECT_ID:
    raise ValueError('Enter a Google Cloud Project ID. Do not enter a password, token, or key.')
ee.Authenticate()  # Sign in with your Google account when Colab prompts you.
ee.Initialize(project=EE_PROJECT_ID)
print('Earth Engine initialized.')

## Choose a Lagos boundary

The default uses FAO GAUL 2025 second-level administrative boundaries. Review the printed names and select one LGA to keep the query and export manageable.

In [ ]:
admin2 = ee.FeatureCollection('FAO/GAUL/2025/level2')
lagos_areas = (admin2
    .filter(ee.Filter.eq('GAUL0_NAME', 'Nigeria'))
    .filter(ee.Filter.stringContains('GAUL1_NAME', 'Lagos')))
area_names = sorted(set(lagos_areas.aggregate_array('GAUL2_NAME').getInfo()))
print('Lagos second-level areas:', area_names)
AREA_NAME = input('Area name (press Enter for Ikeja): ').strip() or 'Ikeja'
selected = lagos_areas.filter(ee.Filter.eq('GAUL2_NAME', AREA_NAME))
if selected.size().getInfo() != 1:
    raise ValueError(f'{AREA_NAME!r} did not match one boundary. Choose an exact name from the printed list.')
boundary = ee.Feature(selected.first())
aoi = boundary.geometry()
AREA_KM2 = aoi.area(1).divide(1e6).getInfo()
print('Selected:', AREA_NAME, '| area km²:', round(AREA_KM2, 2))

In [ ]:
# Count and preview the filtered footprint layer before exporting.
MIN_CONFIDENCE = float(input('Minimum confidence from 0.00 to 1.00 (default 0.75): ').strip() or '0.75')
if not 0 <= MIN_CONFIDENCE <= 1 or round(MIN_CONFIDENCE, 2) != MIN_CONFIDENCE:
    raise ValueError('Choose a confidence cutoff from 0.00 to 1.00 in 0.01 increments.')
source_buildings = (ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons')
    .filterBounds(aoi)
    .filter(ee.Filter.gte('confidence', MIN_CONFIDENCE)))
building_count = source_buildings.size().getInfo()
print(f'{building_count:,} footprints at confidence >= {MIN_CONFIDENCE:.2f}')

# Paint server-side map tiles; this previews the features without downloading them to Colab.
painted = ee.Image().byte().paint(source_buildings, 1).selfMask().clip(aoi).visualize(palette=['00A86B'])
tile_url = painted.getMapId()['tile_fetcher'].url_format
center = aoi.centroid(1).coordinates().getInfo()
preview = folium.Map(location=[center[1], center[0]], zoom_start=12, control_scale=True)
folium.TileLayer(tiles=tile_url, attr='Google Earth Engine / Google Open Buildings', name='Open Buildings V3', overlay=True).add_to(preview)
folium.LayerControl().add_to(preview)
display(preview)


In [ ]:
# Export only the clipped footprint features.
from datetime import datetime, timezone

if 'MIN_CONFIDENCE' not in globals():
    MIN_CONFIDENCE = float(input('Minimum confidence from 0.00 to 1.00 (default 0.75): ').strip() or '0.75')
if not 0 <= MIN_CONFIDENCE <= 1 or round(MIN_CONFIDENCE, 2) != MIN_CONFIDENCE:
    raise ValueError('Choose a confidence cutoff from 0.00 to 1.00 in 0.01 increments.')

source_buildings = (ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons')
    .filterBounds(aoi)
    .filter(ee.Filter.gte('confidence', MIN_CONFIDENCE)))
building_count = source_buildings.size().getInfo()
print(f'{building_count:,} footprints at confidence >= {MIN_CONFIDENCE:.2f}')

def clip_building(feature):
    feature = ee.Feature(feature)
    clipped_geometry = feature.geometry().intersection(aoi, ee.ErrorMargin(1))
    return ee.Feature(clipped_geometry).set({
        'confidence': feature.get('confidence'),
        'clipped_area_m2': clipped_geometry.area(1)
    })

export_features = source_buildings.map(clip_building)
threshold_tag = f'{MIN_CONFIDENCE:.2f}'.replace('.', '')
description = f'{AREA_NAME}_open_buildings_conf{threshold_tag}'
EXPORT_STARTED_UTC = datetime.now(timezone.utc).isoformat()
geojson_task = ee.batch.Export.table.toDrive(
    collection=export_features,
    description=description,
    folder='Lagos_Open_Buildings',
    fileNamePrefix=description,
    fileFormat='GeoJSON')
geojson_task.start()
print('Footprint export started:', geojson_task.status())


## Outputs, repeat runs, and attribution

The last code cell exports the small boundary GeoJSON and one-row CSV summary alongside the footprint GeoJSON in `My Drive/Lagos_Open_Buildings/`:

- `<LGA>_open_buildings_conf<TAG>.geojson` — clipped footprint features and model confidence;
- `<LGA>_boundary_gaul2025.geojson` — the selected FAO GAUL 2025 boundary used for clipping;
- `<LGA>_open_buildings_conf<TAG>_summary.csv` — area, cutoff, feature count, source, license, and export timestamp.

After the first run, rerun only the final code cell to refresh the small boundary and summary exports without exporting the footprint GeoJSON again. In a fresh runtime, run the authentication and boundary-selection cells first, then run the final cell; it will count matching predictions on Earth Engine without downloading their geometries.

The feature GeoJSON retains `confidence` and `clipped_area_m2`. The count is model predictions, not a complete building register or an accuracy score. Credit Google Research Open Buildings V3 and FAO GAUL 2025 under CC BY 4.0.


In [ ]:
# Export the selected boundary and summary metadata (safe to rerun without repeating the footprint export).
from datetime import datetime, timezone

if 'MIN_CONFIDENCE' not in globals():
    MIN_CONFIDENCE = float(input('Minimum confidence from 0.00 to 1.00 (default 0.75): ').strip() or '0.75')
if not 0 <= MIN_CONFIDENCE <= 1 or round(MIN_CONFIDENCE, 2) != MIN_CONFIDENCE:
    raise ValueError('Choose a confidence cutoff from 0.00 to 1.00 in 0.01 increments.')
if 'building_count' not in globals():
    source_buildings = (ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons')
        .filterBounds(aoi)
        .filter(ee.Filter.gte('confidence', MIN_CONFIDENCE)))
    building_count = source_buildings.size().getInfo()

threshold_tag = f'{MIN_CONFIDENCE:.2f}'.replace('.', '')
description = f'{AREA_NAME}_open_buildings_conf{threshold_tag}'
export_started_utc = globals().get('EXPORT_STARTED_UTC', datetime.now(timezone.utc).isoformat())
boundary_features = ee.FeatureCollection([ee.Feature(aoi).set({
    'boundary_name': AREA_NAME,
    'boundary_source': 'FAO GAUL 2025 level 2'
})])
boundary_task = ee.batch.Export.table.toDrive(
    collection=boundary_features,
    description=f'{AREA_NAME}_boundary_gaul2025',
    folder='Lagos_Open_Buildings',
    fileNamePrefix=f'{AREA_NAME}_boundary_gaul2025',
    fileFormat='GeoJSON')
summary = ee.FeatureCollection([ee.Feature(None, {
    'boundary_name': AREA_NAME,
    'boundary_area_km2': AREA_KM2,
    'boundary_source': 'FAO GAUL 2025 level 2',
    'footprint_dataset': 'Google Research Open Buildings V3',
    'source_inference_year': 2023,
    'minimum_confidence': MIN_CONFIDENCE,
    'filtered_footprint_count': building_count,
    'export_started_utc': export_started_utc,
    'source_license': 'CC BY 4.0'
})])
summary_task = ee.batch.Export.table.toDrive(
    collection=summary,
    description=f'{description}_summary',
    folder='Lagos_Open_Buildings',
    fileNamePrefix=f'{description}_summary',
    fileFormat='CSV')
boundary_task.start()
summary_task.start()
tasks = [boundary_task, summary_task]
if 'geojson_task' in globals():
    tasks.append(geojson_task)
print('Waiting for boundary, summary, and any active footprint export...')
while any(task.active() for task in tasks):
    print(' | '.join(f'{task.status().get("description", "export")}: {task.status().get("state")}' for task in tasks))
    time.sleep(20)
statuses = [task.status() for task in tasks]
for status in statuses:
    print(status)
if any(status.get('state') != 'COMPLETED' for status in statuses):
    raise RuntimeError(f'One or more Earth Engine exports failed: {statuses}')
print(f'Finished. Outputs are in My Drive/Lagos_Open_Buildings/: {description}.geojson, {AREA_NAME}_boundary_gaul2025.geojson, {description}_summary.csv')
